In [ ]:
import os
import tempfile
import psutil

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(tempfile.gettempdir(), "xdg-cache"))
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)
os.makedirs(os.environ["XDG_CACHE_HOME"], exist_ok=True)

pid = os.getpid()
print("PID:", pid)

p = psutil.Process(pid)
print("Name:", p.name())
print("Status:", p.status())
try:
    memory = p.memory_full_info()
except psutil.Error:
    memory = p.memory_info()
print("Memory:", memory)


# Set Configuration

In [ ]:
import math
from diffusion_hash_inv.config import MainConfig, HashConfig, MessageConfig, OutputConfig, Byte2RGBConfig
from diffusion_hash_inv.main import MainEP
from diffusion_hash_inv.utils import FileIO
from diffusion_hash_inv.main import RuntimeConfig

sqrt_len = 4
length = sqrt_len * sqrt_len * 8
IMAGES_PER_LOG_ESTIMATE = 71
iteration = 10_000
RUN = True

main_cfg = MainConfig(
    verbose_flag=False,
    clean_flag=True,
    debug_flag=False,
    make_image_flag=True,
)
hash_cfg = HashConfig(
    hash_alg="md5",
    length=length,
)
message_cfg = MessageConfig(
    message_flag=False,
    length=length,
    random_flag=True,
    seed_flag=True,
)
output_cfg = OutputConfig()
byte2rgb_cfg = Byte2RGBConfig()
runtime_cfg = RuntimeConfig(
    main=main_cfg,
    message=message_cfg,
    hash=hash_cfg,
    output=output_cfg,
    rgb=byte2rgb_cfg,
)

io_controller = FileIO(main_config=main_cfg, output_cfg=output_cfg)
print(f"Iterations: {iteration} (~{iteration * IMAGES_PER_LOG_ESTIMATE:,} images)")


In [ ]:
if RUN:
    _batch_size = 1000
    assert iteration % _batch_size == 0, "Iteration must be divisible by batch size"
    _batch_num = iteration // _batch_size

    for _ in range(_batch_num):
        print(f"Batch {_+1}/{_batch_num}")
        main = MainEP(runtime_config=runtime_cfg, file_controller=io_controller)
        main.run(iteration=_batch_size)

In [ ]:
from diffusion_hash_inv.logger.logger import Logs
from diffusion_hash_inv.analyze import BetaScheduleAnalyzer


def stream_logs():
    return Logs.iter_logs(io_controller, hash_cfg, main_cfg)


log_count = sum(1 for _ in stream_logs())
print(f"Logs available: {log_count}")


In [ ]:
beta_analyzer = BetaScheduleAnalyzer(step_name="4th Step")


def stream_schedules():
    return beta_analyzer.iter_schedules(stream_logs())


# Beta Scheduling
## Approach 1
**method**  
1. Cumulative all bytes from Hash algorithm calculation logs  
2. Make new Beta Schedule by multiply Base Beta Schedule and Cumulative block


In [ ]:
summary = BetaScheduleAnalyzer.summarize(stream_schedules())
assert summary.count == log_count, f"Step 4 logs mismatch: {summary.count} != {log_count}"
print(f"Schedules: {summary.count}")
print(f"Schedule length: {summary.length}")


In [ ]:
# Step 4 schedules are streamed by diffusion_hash_inv.analyze.BetaScheduleAnalyzer.
# The notebook keeps only online aggregate statistics.


In [ ]:
# Individual schedules are not retained by the streaming summary.


In [ ]:
print(f"Summary count: {summary.count}")
print(f"Summary length: {summary.length}")
print(f"Mean preview: {summary.mean[:16]}")


In [ ]:
if main_cfg.verbose_flag:
    print(f"Schedules summarized: {summary.count}")
    print(f"Mean preview: {summary.mean[:64]}")


In [ ]:
import numpy as np
from diffusion_hash_inv.scheduling import BetaScheduler

beta_scheduler = BetaScheduler(beta_min=1e-4, beta_max=2e-2)
betas = beta_scheduler.base_betas(summary.length)


In [ ]:
_min = summary.minimum
_max = summary.maximum
mean = summary.mean
var = summary.variance
std = summary.std

approach1 = beta_scheduler.approach1(mean, base_betas=betas)
beta_candidate_from_multiply = approach1.raw_candidate
beta_candidate_rescaled = approach1.rescaled_candidate

np.set_printoptions(threshold=32, linewidth=160)

ori_beta_min = beta_scheduler.beta_min
ori_beta_max = beta_scheduler.beta_max

print(f"SN shape: ({summary.count}, {summary.length})")
print(f"Min preview: {_min[:8]}")
print(f"Max preview: {_max[:8]}")
print(f"Mean preview: {mean[:8]}")
print(f"Variance preview: {var[:8]}")
print(f"Standard deviation preview: {std[:8]}")


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(20, 10))
plt.plot(_min, label='Min', color='blue')
plt.legend()
plt.title('Min Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(_max, label='Max', color='orange')
plt.legend()
plt.title('Max Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(mean, label='Mean', color='green')
plt.legend()
plt.title('Mean Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(var, label='Variance', color='red')
plt.legend()
plt.title('Variance Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(std, label='Standard Deviation', color='purple')
plt.legend()
plt.title('Standard Deviation Values of SN')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(betas, label='Betas', color='blue')
plt.legend()
plt.title('Original Betas')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
# First schedule plot removed because individual schedules are not retained.

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(beta_candidate_from_multiply, label='Candidate Betas', color='cyan')
plt.legend()
plt.title('Candidate Betas from Multiplication')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

In [ ]:
# Approach 1 rescaled candidate is computed by BetaScheduler.approach1.
print(beta_candidate_rescaled)

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(beta_candidate_rescaled, label='Candidate Betas', color='cyan')
plt.legend()
plt.title('Candidate Betas from Multiplication (Rescaled)')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()

# Beta Scheduling
## Approach 2
**method**  
1. Cumulative all bytes from Hash algorithm calculation logs  
2. Make new Beta scheduling with the cumulative sum of log bytes from intermediate computations on the x-axis  


In [ ]:
approach2 = beta_scheduler.approach2(mean)
sn_min = approach2.sn_min
sn_max = approach2.sn_max
linear_eq_slope = approach2.slope

print(f"Original Beta Min: {ori_beta_min}")
print(f"Original Beta Max: {ori_beta_max}")
print(f"SN Min: {sn_min}")
print(f"SN Max: {sn_max}")
print(f"Linear Equation Slope: {linear_eq_slope}")

In [ ]:
beta_candidate_from_linear_eq = approach2.candidate
print(beta_candidate_from_linear_eq)

In [ ]:
delta = betas - beta_candidate_from_linear_eq
print(f"Max abs delta: {np.max(np.abs(delta))}")
print(f"Delta preview: {delta[:16]}")


In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(beta_candidate_from_linear_eq, label='Make beta schedule using linear equation', color='cyan')
plt.legend()
plt.title('Candidate Betas from Linear Equation')
plt.xlabel('Loop')
plt.ylabel('Values')
plt.show()